<a href="https://colab.research.google.com/github/SanthiniRa/AILearningGD/blob/main/chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

First step is add the openAI

common input and ouput

In [ ]:
user_input = input("How are you feeling today? ")
print("I understand. Tell me more about that.")

How are you feeling today? happy
I understand. Tell me more about that.


Load a Free AI Model

In [ ]:
!pip install transformers torch accelerate

Load DialoGPT-medium from Hugging Face:

In [ ]:
from transformers import pipeline

chatbot = pipeline("text-generation", model="microsoft/DialoGPT-medium")
#chatbot = pipeline("conversational", model="microsoft/DialoGPT-small")

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
Test generating responses:

In [ ]:
def therapist_ai(user_input):
    prompt = f"Therapist: Respond empathetically.\nUser: {user_input}\nTherapist:"
    response = chatbot(prompt, max_length=150, do_sample=True, temperature=0.7)
    return response[0]['generated_text'].split("Therapist:")[-1]

print(therapist_ai("I feel stressed about work."))

Passing `generation_config` together with generation-related arguments=({'max_length', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 It's ok, I think you're doing great. RespondTherapist : Thank you.


In [ ]:
prompt_test = f"Therapist: Respond empathetically.\nUser: {user_input}\nTherapist:"
response_test = chatbot(prompt_test, max_length=150, do_sample=True, temperature=0.7)
print(response_test)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'Therapist: Respond empathetically.\nUser: happy\nTherapist: Happy'}]


In [ ]:
while True:
    user = input("Hi. How are you feeling: ")
    if user.lower() == "quit":
        break
    print("Therapist:", therapist_ai(user))

You: sleepy


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Therapist:  The patient's heart rate has been reduced to 0
You: worried


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Therapist:  I am concerned for your safety
You: quit


In [ ]:
system_prompt = """
You are a calm, empathetic therapist.
Listen carefully, validate feelings, and ask gentle reflective questions.
Do not give medical diagnosis.
"""
def therapist_ai(user_input):
    # Enhanced prompt to encourage more empathetic and detailed responses
    prompt = f"{system_prompt}\nUser: {user_input}\nTherapist: I hear you. It sounds like you're going through a lot. Can you tell me more about what's making you feel this way?"
    response = chatbot(prompt, max_new_tokens=100, do_sample=True, temperature=0.7)
    full_generated_text = response[0]['generated_text']

    # Extract only the part after the prompt (including the example response for better parsing)
    # The model might continue from the example, or start a new response
    expected_start_of_response = "Therapist: I hear you. It sounds like you're going through a lot. Can you tell me more about what's making you feel this way?"

    if full_generated_text.startswith(prompt):
        # If the model continues from the example, extract from there
        if full_generated_text.startswith(prompt + expected_start_of_response):
            print(full_generated_text)
            therapist_response = full_generated_text[len(prompt + expected_start_of_response):].strip()
        else:
            # If it rewrites the therapist part, find the last 'Therapist:'
            last_therapist_idx = full_generated_text.rfind("Therapist:")
            if last_therapist_idx != -1:
                therapist_response = full_generated_text[last_therapist_idx + len("Therapist:"):].strip()
            else:
                therapist_response = full_generated_text[len(prompt):].strip()
    else:
        # Fallback if model doesn't perfectly reproduce the prompt, find last 'Therapist:'
        last_therapist_idx = full_generated_text.rfind("Therapist:")
        if last_therapist_idx != -1:
            therapist_response = full_generated_text[last_therapist_idx + len("Therapist:"):].strip()
        else:
            therapist_response = full_generated_text.strip()

    return therapist_response if therapist_response else "I'm sorry, I couldn't generate a clear response. Can you tell me more?"

while True:
    user = input("How are You feeling: ")
    if user.lower() == "quit":
        break
    print("Therapist:", therapist_ai(user))

How are You feeling: sad


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Therapist: I hear you. It sounds like you're going through a lot. Can you tell me more about what's making you feel this way?
